# Post 008 — Naive Bayes & LDA: Generative Classifiers
## Dataset A: Jira Bug Ticket Routing

**AI Engineering Lab Series | Era 1: Classic Machine Learning**

---

Every engineering team has a bug tracker. Every day, new tickets arrive and someone has to decide: is this a hardware issue, a firmware bug, a software defect, a performance problem, a security vulnerability, or a documentation gap?

This notebook uses **Gaussian Naive Bayes** and **Linear Discriminant Analysis (LDA)** to automatically route bug tickets to the correct team based on ticket features. The key insight: these are **generative models** — instead of learning a decision boundary directly, they learn the probability distribution of each class and use Bayes' theorem to classify.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

## 1. Load and Explore the Dataset

Our dataset contains 8,000 bug ticket records with 10 numerical features extracted from ticket metadata: priority score, number of comments, time to first response, number of linked tickets, reporter seniority, component count, log file size, reproduction rate, customer impact score, and days since last similar bug.

In [ ]:
df = pd.read_csv('../data/jira_bug_routing.csv')
print(f'Shape: {df.shape}')
print(f'\nTeam distribution:')
print(df['team'].value_counts())
df.head()

In [ ]:
# Visualize feature distributions per team
feature_cols = [c for c in df.columns if c != 'team']
teams = df['team'].unique()

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()
colors = plt.cm.Set1(np.linspace(0, 1, len(teams)))

for i, feat in enumerate(feature_cols):
    for team, col in zip(teams, colors):
        data = df.loc[df['team']==team, feat]
        axes[i].hist(data, bins=20, alpha=0.5, color=col, label=team, density=True)
    axes[i].set_title(feat, fontsize=8)
    axes[i].tick_params(labelsize=7)

axes[0].legend(fontsize=6, loc='upper right')
plt.suptitle('Feature Distributions per Team (Jira Bug Routing)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Understanding Bayes' Theorem

Naive Bayes classifies by computing:

$$P(team | features) \propto P(features | team) \times P(team)$$

- **P(team)** = Prior: how common is each team's ticket? (from training data frequency)
- **P(features | team)** = Likelihood: given this team, how likely are these feature values?
- **P(team | features)** = Posterior: the final classification probability

The 'Naive' part: we assume all features are **conditionally independent** given the class. This is rarely true in practice, but the model is surprisingly robust despite this assumption.

In [ ]:
# Prepare data
le = LabelEncoder()
X = df[feature_cols].values
y = le.fit_transform(df['team'].values)
class_names = le.classes_

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Show class priors
gnb = GaussianNB()
gnb.fit(X_train_scaled, y_train)

print('Class Priors (P(team)):')  
for name, prior in zip(class_names, gnb.class_prior_):
    print(f'  {name:20s}: {prior:.3f}')

## 3. Gaussian Naive Bayes

Gaussian NB assumes each feature follows a normal distribution within each class. It learns the mean and variance of each feature per class during training. Classification then computes the Gaussian likelihood for each class and picks the highest posterior.

In [ ]:
# Evaluate Gaussian NB
y_pred_gnb = gnb.predict(X_test_scaled)
y_prob_gnb = gnb.predict_proba(X_test_scaled)

print('Gaussian Naive Bayes Results:')
print(f'Accuracy: {accuracy_score(y_test, y_pred_gnb):.3f}')
print()
print(classification_report(y_test, y_pred_gnb, target_names=class_names))

# Cross-validation
cv_scores = cross_val_score(gnb, X_train_scaled, y_train, cv=StratifiedKFold(5), scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_gnb)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Gaussian NB: Confusion Matrix (Jira Bug Routing)')
ax.set_xlabel('Predicted Team')
ax.set_ylabel('True Team')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 4. Linear Discriminant Analysis (LDA)

LDA is a close cousin of Naive Bayes. Both are generative models, but LDA makes a stronger assumption: all classes share the **same covariance matrix**. In return, LDA can also be used for **dimensionality reduction** — it finds the linear combinations of features that best separate the classes.

This dual role (classifier + dimensionality reducer) makes LDA uniquely powerful for high-dimensional problems.

In [ ]:
# LDA as classifier
lda = LinearDiscriminantAnalysis()
lda.fit(X_train_scaled, y_train)
y_pred_lda = lda.predict(X_test_scaled)

print('LDA Classifier Results:')
print(f'Accuracy: {accuracy_score(y_test, y_pred_lda):.3f}')
print()
print(classification_report(y_test, y_pred_lda, target_names=class_names))

cv_lda = cross_val_score(lda, X_train_scaled, y_train, cv=StratifiedKFold(5), scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv_lda.mean():.3f} ± {cv_lda.std():.3f}')

In [ ]:
# LDA as dimensionality reducer — visualize the discriminant space
lda_2d = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda_2d.fit_transform(X_train_scaled, y_train)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# LDA space
for i, (name, col) in enumerate(zip(class_names, colors)):
    mask = y_train == i
    ax1.scatter(X_lda[mask, 0], X_lda[mask, 1], c=[col], label=name, alpha=0.5, s=20)
ax1.set_title('LDA Discriminant Space (Training Data)')
ax1.set_xlabel('LD1'); ax1.set_ylabel('LD2')
ax1.legend(fontsize=8)

# Feature coefficients for LD1 and LD2
coef_df = pd.DataFrame(lda_2d.scalings_[:, :2], index=feature_cols, columns=['LD1', 'LD2'])
coef_df.plot(kind='bar', ax=ax2, color=['steelblue', 'coral'])
ax2.set_title('LDA Feature Coefficients: What drives each discriminant?')
ax2.set_xlabel('Feature')
ax2.set_ylabel('Coefficient')
ax2.axhline(y=0, color='black', linewidth=0.5)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 5. Gaussian NB vs LDA: Head-to-Head Comparison

Both models are fast, interpretable, and work well with small datasets. The key difference is the covariance assumption — Gaussian NB assumes diagonal covariance (features independent per class), LDA assumes shared full covariance across all classes.

In [ ]:
# Side-by-side confusion matrices
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

cm_gnb = confusion_matrix(y_test, y_pred_gnb, normalize='true')
cm_lda = confusion_matrix(y_test, y_pred_lda, normalize='true')

sns.heatmap(cm_gnb, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax1)
ax1.set_title(f'Gaussian NB (Acc={accuracy_score(y_test, y_pred_gnb):.3f})')
ax1.set_xlabel('Predicted'); ax1.set_ylabel('True')
plt.setp(ax1.get_xticklabels(), rotation=30, ha='right')

sns.heatmap(cm_lda, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=ax2)
ax2.set_title(f'LDA (Acc={accuracy_score(y_test, y_pred_lda):.3f})')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')
plt.setp(ax2.get_xticklabels(), rotation=30, ha='right')

plt.suptitle('Normalized Confusion Matrices: Gaussian NB vs LDA', fontsize=13)
plt.tight_layout()
plt.show()

print('\nSummary:')
print(f'Gaussian NB accuracy: {accuracy_score(y_test, y_pred_gnb):.3f}')
print(f'LDA accuracy:         {accuracy_score(y_test, y_pred_lda):.3f}')
print(f'\nLDA also reduces to {min(len(class_names)-1, X.shape[1])} discriminant dimensions')
print('LDA is both a classifier AND a dimensionality reducer')

## 6. Summary

| Model | Assumption | Covariance | Also Reduces Dims? | Best For |
|---|---|---|---|---|
| **Gaussian NB** | Features independent per class | Diagonal per class | No | Text, fast baseline, small data |
| **LDA** | Shared covariance across classes | Full, shared | Yes (n_classes-1) | When classes are well-separated, visualization |

**Key takeaways:**
1. **Naive Bayes is not naive** — despite the independence assumption, it often performs competitively with more complex models
2. **LDA's dual role** as classifier and dimensionality reducer makes it uniquely valuable for understanding class structure
3. **Priors matter** — if your training data has class imbalance, NB will naturally account for it via the prior
4. **Both models are fast** — training on 8,000 samples takes milliseconds, making them ideal for real-time routing systems